# 01 - Exploratory Data Analysis & Data Quality Audit

**Project:** Time-Aware Air Quality Sensor Calibration: CO Estimation and Drift Evaluation  
**Issue:** #2 — EDA & Data Quality  
**Author:** Emran Yaseen Al-Sabry (`EmranYaseen-ALSabry`)  
**Data Contract Version:** v1.1 (`docs/data_contract.md`)

---

## Objective & Scientific Principles
This notebook conducts a focused, descriptive exploratory data analysis (EDA) and rigorous data quality audit on the UCI Air Quality dataset.
In strict alignment with the project foundation:
1. **Strict Zero Data Leakage:** All exploratory investigations relating low-cost sensor inputs to the reference target `CO(GT)` utilize **only the Development Period** (`< TEST_START = 2005-01-17 00:00`). The final holdout test set remains completely isolated.
2. **Official Loader Utilization:** We load and validate the data using the contract loader from `air_quality.data.load_raw()`.
3. **Reproducibility & Measured Figures:** High-resolution figures are exported to `reports/figures/eda/` to document missingness patterns, target dynamics, sensor responsiveness, and temporal stability.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Ensure local source package is accessible
sys.path.insert(0, str(Path.cwd().parent / "src" if Path.cwd().name == "notebooks" else Path.cwd() / "src"))

from air_quality.data import (
    load_raw,
    TARGET,
    SENSOR_COLS,
    ENV_COLS,
    INPUT_COLS,
    FORBIDDEN_COLS,
    NUMERIC_COLS,
    inputs_available,
    eligible_mask,
    validate,
)
from air_quality.split import split_dev_test, TEST_START, DATA_START

# Matplotlib styling for publication quality
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "figure.titlesize": 13,
})

def numpy_kde(data: np.ndarray, x_grid: np.ndarray, bandwidth: float | None = None) -> np.ndarray:
    # Pure NumPy 1D Gaussian Kernel Density Estimation (Silverman's rule of thumb)
    data = np.asarray(data, dtype=float)
    n = len(data)
    if bandwidth is None:
        bandwidth = 1.06 * float(np.std(data, ddof=1)) * (n ** -0.2)
    u = (x_grid[:, None] - data[None, :]) / bandwidth
    return np.sum(np.exp(-0.5 * u**2) / (np.sqrt(2.0 * np.pi) * bandwidth), axis=1) / n

FIG_DIR = Path.cwd().parent / "reports" / "figures" / "eda" if Path.cwd().name == "notebooks" else Path("reports/figures/eda")
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f"Setup complete. Figures will be saved to: {FIG_DIR}")


### Verification of Configuration and Tooling
The environment has been initialized with the verified project interface from `air_quality.data` and `air_quality.split`. The target variable is `CO(GT)` and the input variables comprise 5 metal-oxide sensor readings and 3 environmental features (`T`, `RH`, `AH`). All reference analyzer features (`FORBIDDEN_COLS`) are strictly excluded from predictive workflows.


In [ ]:
# Ingest data using the verified contract loader
df = load_raw()
dev_df, test_df = split_dev_test(df)

validation_report = validate(df)
print("=== Complete Hourly Grid Validation Report ===")
for k, v in validation_report.items():
    if k != "missing_per_column":
        print(f"{k:32s}: {v}")

print(f"\nTotal Dataset Shape : {df.shape}")
print(f"Development Period  : {dev_df.index.min()} to {dev_df.index.max()} (Hours: {len(dev_df):,})")
print(f"Final Test Period   : {test_df.index.min()} to {test_df.index.max()} (Hours: {len(test_df):,})")


### Analysis of Hourly Grid Integrity
The loaded dataset spans **9,357 continuous hourly observations** starting at `2004-03-10 18:00:00` and ending at `2005-04-04 14:00:00`.
- The temporal index contains **0 missing grid hours** and **0 duplicate timestamps**.
- The chronological split cleanly separates the timeline at `TEST_START = 2005-01-17 00:00:00` into:
  - **Development set:** 7,494 hours (80.09% of grid).
  - **Final test holdout:** 1,863 hours (19.91% of grid).


In [ ]:
# 1. Missing values per column
missing_table = pd.DataFrame({
    "Missing Hours": df.isna().sum(),
    "Missing %": (df.isna().mean() * 100).round(2)
})
display(missing_table)

# 2. Analyze continuous outage runs (missing runs)
def calculate_missing_runs(series: pd.Series) -> pd.Series:
    is_na = series.isna()
    blocks = (~is_na).cumsum()
    runs = is_na.groupby(blocks).sum()
    return runs[runs > 0]

target_runs = calculate_missing_runs(df[TARGET])
input_outages = calculate_missing_runs(pd.Series(np.where(~inputs_available(df), np.nan, 1), index=df.index))

print(f"Target CO(GT) total missing runs count: {len(target_runs)}")
print(f"Longest continuous target outages (hours): {target_runs.nlargest(5).tolist()}")
print(f"\nLow-cost sensor input outages count: {len(input_outages)}")
print(f"Longest continuous sensor outages (hours): {input_outages.nlargest(5).tolist()}")


### Analysis of Missingness Architecture
1. **Low-Cost Inputs: All-or-Nothing Synchronous Outages:**
   All 5 low-cost sensors and all 3 environmental readings are missing simultaneously in exactly **366 hours** (3.91% of timeline). There are **0 partial hours** where some sensors were online while others were offline. The longest continuous low-cost sensor outage lasts **76 consecutive hours** (observed in February 2005 during the test period).
2. **Reference Analyzer Outages:**
   The reference analyzer `CO(GT)` is missing in **1,683 hours** (17.99%) across **187 discrete outage episodes**. Several multi-day gaps occur: 173h (October 2004), 146h (April 2004), 142h (October 2004), 122h (late July 2004), and 96h (late December 2004).


In [ ]:
# Figure 1: Missingness Timeline Across All Variables
fig, ax = plt.subplots(figsize=(14, 6), dpi=300)

cols_to_plot = [TARGET, *SENSOR_COLS, *ENV_COLS, "NOx(GT)", "NO2(GT)", "C6H6(GT)", "NMHC(GT)"]
missing_matrix = df[cols_to_plot].isna().T.astype(int)

ax.imshow(
    missing_matrix,
    aspect="auto",
    cmap="Blues",
    interpolation="nearest",
    extent=[mdates.date2num(df.index[0]), mdates.date2num(df.index[-1]), len(cols_to_plot) - 0.5, -0.5],
)

ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax.set_yticks(range(len(cols_to_plot)))
ax.set_yticklabels(cols_to_plot, fontsize=9)
ax.axvline(mdates.date2num(TEST_START), color="red", linestyle="--", linewidth=2, label="TEST_START (2005-01-17)")

ax.annotate("Target Outage (173h)", xy=(mdates.date2num(pd.Timestamp("2004-10-13")), 0),
            xytext=(mdates.date2num(pd.Timestamp("2004-09-01")), -0.8),
            arrowprops=dict(facecolor="black", arrowstyle="->", lw=1), fontsize=9, fontweight="bold")

ax.annotate("All-Sensors Outage (76h)", xy=(mdates.date2num(pd.Timestamp("2005-02-10")), 3),
            xytext=(mdates.date2num(pd.Timestamp("2005-01-01")), 1.5),
            arrowprops=dict(facecolor="darkblue", arrowstyle="->", lw=1), fontsize=9, fontweight="bold", color="darkblue")

ax.set_title("Figure 1: Missingness Timeline Across All Variables (March 2004 - April 2005)\n(Blue = Missing / NaN)", pad=15)
ax.set_xlabel("Timeline")
ax.legend(loc="lower right")
plt.tight_layout()
fig1_path = FIG_DIR / "01_missingness_timeline.png"
fig.savefig(fig1_path)
plt.show()


### Interpretation of Figure 1 and Eligibility Partitioning
Figure 1 clearly illustrates the structural difference in data quality across time.
- The reference analyzer (first row) shows several dense clusters of outages during Summer and Autumn 2004.
- During early 2005 (the test period), the reference analyzer was online almost continuously.
- **Eligibility breakdown (per `docs/data_contract.md` §7):**
  - Full Dataset: **7,344** eligible hours (78.49%).
  - Development Period: **5,657** eligible hours (75.49%).
  - Final Test Period: **1,687** eligible hours (90.55%).


In [ ]:
# Evaluate target distribution on development period only
dev_eligible = eligible_mask(dev_df)
dev_co = dev_df.loc[dev_eligible, TARGET]

co_summary = pd.Series({
    "Count": len(dev_co),
    "Mean (mg/m³)": dev_co.mean(),
    "Std (mg/m³)": dev_co.std(),
    "Median (mg/m³)": dev_co.median(),
    "IQR (mg/m³)": dev_co.quantile(0.75) - dev_co.quantile(0.25),
    "95th Percentile (mg/m³)": dev_co.quantile(0.95),
    "Min (mg/m³)": dev_co.min(),
    "Max (mg/m³)": dev_co.max(),
    "Skewness": dev_co.skew()
})
print("=== Reference CO(GT) Development Distribution Statistics ===")
display(co_summary.to_frame("Statistic"))

# Monthly statistics
dev_monthly = dev_df[dev_eligible].groupby(pd.Grouper(freq="ME"))[TARGET].agg(
    ["count", "mean", "std", "median"]
).round(2)
print("\n=== Monthly CO(GT) Profile across Development Period ===")
display(dev_monthly)


### Target Distribution Characteristics
- The distribution of reference `CO(GT)` is **positively skewed (+1.36)**, with a median of **1.90 mg/m³**, a mean of **2.19 mg/m³**, and a 95th percentile of **4.90 mg/m³**.
- Values range from **0.10 mg/m³** to **11.90 mg/m³**.
- **Monthly Variation:**
  - **August 2004** exhibits the lowest monthly mean (**1.25 mg/m³**, std 0.58).
  - **October and November 2004** exhibit peak levels (**2.72 mg/m³** and **2.69 mg/m³**, std up to 1.93).
  - Any broader meteorological or societal explanations remain unverified hypotheses outside the recorded features.


In [ ]:
# Figure 2: Target Distribution & Monthly Boxplots (compatible with matplotlib 3.8.4)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=300)

# Panel A: Density using pure NumPy KDE
ax1.hist(dev_co, bins=35, density=True, alpha=0.6, color="#2b5c8f", edgecolor="white")
x_vals = np.linspace(0, 12, 200)
kde_vals = numpy_kde(dev_co.values, x_vals)
ax1.plot(x_vals, kde_vals, color="#e24a33", lw=2, label="KDE density")
ax1.axvline(dev_co.mean(), color="#e24a33", linestyle="--", lw=1.5, label=f"Mean: {dev_co.mean():.2f}")
ax1.axvline(dev_co.median(), color="#348abd", linestyle="-.", lw=1.5, label=f"Median: {dev_co.median():.2f}")
ax1.set_title("A: CO(GT) Distribution (Development Period)")
ax1.set_xlabel("CO(GT) [mg/m³]")
ax1.set_ylabel("Density")
ax1.set_xlim(0, 12)
ax1.legend(loc="upper right")

# Panel B: Monthly Boxplots (compatible across all matplotlib versions)
dev_monthly_data = dev_df[dev_eligible].copy()
dev_monthly_data["Month"] = dev_monthly_data.index.strftime("%Y-%m")
month_order = sorted(dev_monthly_data["Month"].unique())
box_data = [dev_monthly_data[dev_monthly_data["Month"] == m][TARGET].dropna() for m in month_order]

bp = ax2.boxplot(box_data, patch_artist=True, showmeans=True,
                 meanprops=dict(marker='o', markeredgecolor='black', markerfacecolor='#e24a33'))
for patch in bp['boxes']:
    patch.set_facecolor('#a6c8e0')
    patch.set_edgecolor('#2b5c8f')

ax2.set_xticks(range(1, len(month_order) + 1))
ax2.set_xticklabels([m[5:] for m in month_order])

ax2.set_title("B: Monthly CO(GT) Variation Across Development Months (2004-03 to 2005-01)")
ax2.set_xlabel("Month (Red circle = Mean, Horizontal bar = Median)")
ax2.set_ylabel("CO(GT) [mg/m³]")
ax2.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
fig2_path = FIG_DIR / "02_target_distribution_and_monthly.png"
fig.savefig(fig2_path)
plt.show()


### Interpretation of Figure 2
The visual inspection confirms the long right tail in the target distribution. The monthly boxplots in Panel B demonstrate significant seasonal dynamics: the interquartile range expands during Autumn/Winter, indicating that a static calibration baseline without seasonal or environmental adaptation will face increased variance during colder periods.


In [ ]:
# Section 3: Sensor-to-Target Calibration Relationship (Development Period Only)
dev_valid = dev_df[dev_eligible].copy()
x_s1 = dev_valid["PT08.S1(CO)"]
y_co = dev_valid[TARGET]
t_temp = dev_valid["T"]

pearson_r = y_co.corr(x_s1, method="pearson")
spearman_rho = y_co.corr(x_s1, method="spearman")
slope, intercept = np.polyfit(x_s1, y_co, 1)

residuals = y_co - (slope * x_s1 + intercept)
warm_mask = t_temp > 25
cold_mask = t_temp < 10

print(f"Sample size (N)              : {len(y_co):,}")
print(f"Pearson Correlation (r)      : {pearson_r:.4f}")
print(f"Coefficient of Determ. (R²)  : {pearson_r**2:.4f}")
print(f"Spearman Rank Corr. (rho)    : {spearman_rho:.4f}")
print(f"OLS Linear Fit               : CO(GT) = {slope:.5f} * PT08.S1(CO) + ({intercept:.3f})")
print(f"\nResidual Analysis by Temperature Band:")
print(f"  Warm hours (T > 25°C, n={warm_mask.sum():,}): Mean residual = {residuals[warm_mask].mean():.3f} mg/m³")
print(f"  Cold hours (T < 10°C, n={cold_mask.sum():,}): Mean residual = {residuals[cold_mask].mean():.3f} mg/m³")


### Physical Significance of Sensor Sensitivity & Temperature Modulation
`PT08.S1(CO)` exhibits a strong linear correlation with `CO(GT)` (**r = 0.8931**, Spearman **rho = 0.8999**, $R^2 \approx 0.7976$).
Residual analysis reveals that cold hours sit modestly above the linear fit (+0.181 mg/m³), while warm hours sit very close to the fit (-0.015 mg/m³). This empirical asymmetry provides motivation for testing temperature compensation terms in downstream feature engineering.


In [ ]:
# Figure 3: Target vs PT08.S1(CO) Relationship & Temperature Effect
fig, ax = plt.subplots(figsize=(9, 6), dpi=300)

scatter = ax.scatter(x_s1, y_co, c=t_temp, cmap="coolwarm", alpha=0.6, s=18, edgecolors="none")
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("Ambient Temperature T [°C]", rotation=270, labelpad=15)

x_line = np.linspace(x_s1.min(), x_s1.max(), 100)
ax.plot(x_line, slope * x_line + intercept, color="black", lw=2, linestyle="--",
        label=f"Linear Fit: y = {slope:.5f}x + {intercept:.3f}")

info_box = (
    f"Pearson r = {pearson_r:.4f}\n"
    f"Spearman ρ = {spearman_rho:.4f}\n"
    f"R² = {pearson_r**2:.4f}\n"
    f"Dev Samples = {len(y_co):,}"
)
ax.text(0.05, 0.78, info_box, transform=ax.transAxes, fontsize=10,
        bbox=dict(boxstyle="round,pad=0.5", facecolor="white", edgecolor="gray", alpha=0.9))

ax.set_title("Figure 3: Sensor Response PT08.S1(CO) vs Reference CO(GT) in Development Period\n(Color-coded by Ambient Temperature)", pad=12)
ax.set_xlabel("PT08.S1(CO) Sensor Response")
ax.set_ylabel("Reference CO(GT) [mg/m³]")
ax.legend(loc="lower right")
plt.tight_layout()
fig3_path = FIG_DIR / "03_target_sensor_relationship_dev.png"
fig.savefig(fig3_path)
plt.show()


### Interpretation of Figure 3
The color-coded scatter plot confirms the primary linear alignment while illustrating subtle temperature modulation: colder points tend to lie slightly above the fitted line, which motivates evaluating temperature interaction terms in downstream experiments.


In [ ]:
# Section 4: Temporal Stability (Rolling Correlation with min_periods thresholds)
# 14-day window: require min 168 valid hours (7 full days)
# 30-day window: require min 360 valid hours (15 full days)
roll_14d = dev_valid[TARGET].rolling("14D", min_periods=168).corr(dev_valid["PT08.S1(CO)"])
roll_30d = dev_valid[TARGET].rolling("30D", min_periods=360).corr(dev_valid["PT08.S1(CO)"])

rolling_stats = pd.DataFrame({
    "14-Day (min 168h)": roll_14d.describe(),
    "30-Day (min 360h)": roll_30d.describe()
}).round(3)
display(rolling_stats)

print(f"14-Day min: {roll_14d.min():.4f} at {roll_14d.idxmin()}")
print(f"30-Day min: {roll_30d.min():.4f} at {roll_30d.idxmin()}")
print(f"November 2004 min 14-day: {roll_14d.loc['2004-11'].min():.4f}, 30-day: {roll_30d.loc['2004-11'].min():.4f}")


### Analysis of Rolling Correlation Stability
With small-sample noise filtered out by requiring `min_periods` (168h for 14d, 360h for 30d):
- The 14-day rolling correlation maintains a mean of **0.914** and median of **0.924** (minimum of **0.801** on 2004-07-26 in late July).
- The 30-day rolling correlation maintains a mean of **0.905** and median of **0.907** (minimum of **0.800** on 2004-08-11 in mid-August).
- During November 2004, the correlation remains strong (minimum **0.855** on 14-day and **0.900** on 30-day).
The coupling between sensor and reference instrument remains remarkably robust across all seasons.


In [ ]:
# Figure 4: Rolling Correlation Stability (with min_periods thresholds)
fig, ax = plt.subplots(figsize=(12, 5), dpi=300)

ax.plot(roll_14d.index, roll_14d, label="14-Day Rolling Pearson r (min 168h)", color="#2b5c8f", lw=1.2, alpha=0.85)
ax.plot(roll_30d.index, roll_30d, label="30-Day Rolling Pearson r (min 360h)", color="#e24a33", lw=2)
ax.axhline(pearson_r, color="black", linestyle="--", lw=1.2, label=f"Overall Dev Pearson r ({pearson_r:.4f})")

ax.set_title("Figure 4: Temporal Stability of Correlation between PT08.S1(CO) and CO(GT)\n(Development Period, min_periods: 168h for 14-day, 360h for 30-day)", pad=12)
ax.set_xlabel("Date (2004-03 to 2005-01)")
ax.set_ylabel("Rolling Pearson Correlation")
ax.set_ylim(0.75, 1.02)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend(loc="lower left")

plt.tight_layout()
fig4_path = FIG_DIR / "04_rolling_correlation_stability.png"
fig.savefig(fig4_path)
plt.show()


### Interpretation of Figure 4
Figure 4 demonstrates that the sensor-to-target coupling is consistently high ($r \ge 0.80$) throughout the entire development timeline. The modest reduction during late July and August coincides with the lowest annual ambient CO concentrations, which naturally compresses signal variance relative to baseline noise.


In [ ]:
# Section 5: Descriptive Sensor & Environmental Shifts (Early vs Late Development Period)
cols_to_compare = SENSOR_COLS + ["T", "AH"]
early_period = dev_df.loc["2004-03-11":"2004-05-11", cols_to_compare].dropna()
late_period = dev_df.loc["2004-11-15":"2005-01-16", cols_to_compare].dropna()

shift_table = pd.DataFrame({
    "Early Mean (Spring '04)": early_period.mean(),
    "Late Mean (Winter '04-'05)": late_period.mean(),
    "Absolute Shift": late_period.mean() - early_period.mean(),
    "Percentage Shift (%)": ((late_period.mean() - early_period.mean()) / early_period.mean() * 100).round(2)
})
display(shift_table)

print(f"Early window grid hours: {len(dev_df.loc['2004-03-11':'2004-05-11']):,} (Complete rows: {len(early_period):,})")
print(f"Late window grid hours : {len(dev_df.loc['2004-11-15':'2005-01-16']):,} (Complete rows: {len(late_period):,})")


### Descriptive Evaluation of Sensor Distribution Shifts
Comparing the early window (1,488 grid hours, 1,461 complete rows) to the late window (1,512 grid hours, 1,380 complete rows):
- `PT08.S1(CO)` remains stable (**-2.64%** shift).
- `PT08.S2(NMHC)` shifts slightly (**+2.04%**).
- `PT08.S3(NOx)` drops by **-16.78%**.
- `PT08.S4(NO2)` drops by **-21.18%**.
- `PT08.S5(O3)` increases by **+18.23%**.
- Environmental variables: Temperature drops by **-5.06°C** (from 16.09°C to 11.03°C, -31.45%) and absolute humidity drops from 0.902 to 0.788 (-12.64%).

> **Crucial Scientific Caution:**  
> These shifts **cannot be claimed as proven sensor drift**. In winter, ambient temperatures and atmospheric photochemical dynamics shift substantially. The differences observed combine atmospheric seasonal variation, traffic patterns, and potential sensor surface aging.


In [ ]:
# Figure 5: Sensor Distribution Comparison (Early vs Late) using pure NumPy KDE
fig, axes = plt.subplots(1, 5, figsize=(18, 4), dpi=300, sharey=False)
sensor_names_short = ["S1 (CO)", "S2 (NMHC)", "S3 (NOx)", "S4 (NO2)", "S5 (O3)"]

for idx, col in enumerate(SENSOR_COLS):
    ax = axes[idx]
    e_data = early_period[col].values
    l_data = late_period[col].values
    
    x_min = min(e_data.min(), l_data.min())
    x_max = max(e_data.max(), l_data.max())
    xs = np.linspace(x_min, x_max, 150)
    
    kde_e = numpy_kde(e_data, xs)
    kde_l = numpy_kde(l_data, xs)
    
    ax.plot(xs, kde_e, color="#2b5c8f", lw=2, label="Early Dev (Spring '04)")
    ax.fill_between(xs, kde_e, color="#2b5c8f", alpha=0.2)
    ax.plot(xs, kde_l, color="#e24a33", lw=2, label="Late Dev (Winter '04-'05)")
    ax.fill_between(xs, kde_l, color="#e24a33", alpha=0.2)
    
    pct = ((l_data.mean() - e_data.mean()) / e_data.mean()) * 100
    ax.set_title(f"{sensor_names_short[idx]}\nΔ Mean: {pct:+.1f}%", fontsize=10)
    ax.set_xlabel("Sensor Response")
    if idx == 0:
        ax.set_ylabel("Density")
        ax.legend(loc="upper right", fontsize=8)
    ax.grid(True, linestyle="--", alpha=0.5)

fig.suptitle("Figure 5: Descriptive Comparison of Sensor Distributions (Early vs. Late Development Period)\n*Note: Differences reflect combined atmospheric seasonality and potential aging; drift is not proven.", fontsize=11, y=1.05)
plt.tight_layout()
fig5_path = FIG_DIR / "05_sensor_distribution_early_vs_late.png"
fig.savefig(fig5_path, bbox_inches="tight")
plt.show()


### Interpretation of Figure 5
The visual density plots reveal clear distributional shifts for `PT08.S3(NOx)` and `PT08.S4(NO2)`. While `PT08.S1(CO)` preserves its primary mode, multi-sensor calibration models must learn cross-compensation across these shifting baselines.


In [ ]:
# Section 6: Environmental Trends and Diurnal Traffic Cycles
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), dpi=300)

# Panel A: 24h rolling environmental trend across entire timeline
ax1_twin = ax1.twinx()
p1 = ax1.plot(df.index, df["T"].rolling(24).mean(), color="#e24a33", lw=1.5, label="Temperature T (24h avg) [°C]")
p2 = ax1_twin.plot(df.index, df["AH"].rolling(24).mean(), color="#348abd", lw=1.5, label="Absolute Humidity AH (24h avg)")
ax1.axvline(mdates.date2num(TEST_START), color="black", linestyle="--", lw=1.5, label="TEST_START")

ax1.set_title("A: Environmental Seasonality Across Timeline")
ax1.set_ylabel("Temperature [°C]", color="#e24a33")
ax1_twin.set_ylabel("Absolute Humidity [AH]", color="#348abd")
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=1))

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1_twin.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

# Panel B: Diurnal 24h traffic profile (Summer vs Winter)
dev_df_hourly = dev_df.copy()
dev_df_hourly["Hour"] = dev_df_hourly.index.hour
summer = dev_df_hourly.loc["2004-06-01":"2004-08-31"]
winter = dev_df_hourly.loc["2004-12-01":"2005-01-16"]

summer_valid = summer.loc[dev_eligible.loc[summer.index]]
winter_valid = winter.loc[dev_eligible.loc[winter.index]]
summer_co = summer_valid.groupby(summer_valid.index.hour)[TARGET].mean()
winter_co = winter_valid.groupby(winter_valid.index.hour)[TARGET].mean()

ax2.plot(range(24), summer_co, marker="o", color="#348abd", lw=2, label="CO(GT) Diurnal Mean - Summer")
ax2.plot(range(24), winter_co, marker="s", color="#e24a33", lw=2, label="CO(GT) Diurnal Mean - Winter")
ax2.set_title("B: Diurnal Profile of Reference CO(GT) Concentration by Hour of Day (Summer vs. Winter)")
ax2.set_xlabel("Hour of Day (0 - 23)")
ax2.set_ylabel("Mean CO(GT) [mg/m³]")
ax2.set_xticks(range(24))
ax2.grid(True, linestyle="--", alpha=0.6)
ax2.legend(loc="upper left")

ax2.annotate("Morning rush (8-9 AM)", xy=(8, winter_co.loc[8]), xytext=(9, winter_co.max() * 0.95),
             arrowprops=dict(arrowstyle="->", color="black"), fontsize=9)
ax2.annotate("Evening rush (7-8 PM)", xy=(19, winter_co.loc[19]), xytext=(14, winter_co.max() * 0.95),
             arrowprops=dict(arrowstyle="->", color="black"), fontsize=9)

plt.tight_layout()
fig6_path = FIG_DIR / "06_environmental_seasonal_patterns.png"
fig.savefig(fig6_path)
plt.show()


### Interpretation of Figure 6
The diurnal cycle clearly exhibits roadside bimodal peaks:
1. **Morning Rush Hour (08:00–09:00):** Spike during morning commuting.
2. **Evening Rush Hour (19:00–20:00):** Prominent evening peak.
3. **Mid-day Dip:** Convective atmospheric mixing dilutes surface concentrations.
These cyclical patterns suggest testing cyclical time embeddings ($\sin/\cos$) in Issue #3.


## 7. Synthesis & Downstream Recommendations

### Summary of Audit Findings
1. **Data Completeness & Integrity:** Ingested 9,357 verified hours with 0 missing timestamps or duplicate records.
2. **Missingness:** 366 hours of all-or-nothing sensor outages; 1,683 hours of clustered reference analyzer downtime.
3. **Target Properties:** Moderately right-skewed (+1.36) with seasonal variation (August mean 1.25 vs October mean 2.72 mg/m³).
4. **Calibration Viability:** `PT08.S1(CO)` has strong linear correlation ($r = 0.8931$, $R^2 \approx 0.7976$), with temperature-dependent residual asymmetry (+0.181 cold vs -0.015 warm).
5. **Drift vs Seasonality:** Significant sensor mean shifts between early and late periods are confounded with a 5.06°C temperature drop and seasonal atmospheric chemistry. Drift cannot be claimed as proven.

---

### Ideas to Test for Feature Engineering (Issue #3)
*(Ownership remains with Issue #3)*
- **Sensor-Temperature Interactions:** Test interaction terms such as `PT08.S1 * T` and `PT08.S1 * AH`.
- **Lag Features & Missing Gaps:** Adhere strictly to `docs/data_contract.md` §10 rule 2 (lags falling in missing periods must be NaN; forward-filling is prohibited). Test binary missingness indicators as feature inputs.
- **Cyclical Features:** Test cyclical encodings ($\sin/\cos$) of `Hour` (24h) and `DayOfWeek` (7d).

### Ideas to Test for Modeling & Evaluation (Issue #4)
*(Ownership remains with Issue #4)*
- **Evaluation Metric vs Loss:** While **MAE** is the evaluation metric, consider whether robust loss functions or target transforms handle the right tail.
- **Time-Aware CV:** Evaluate models using the 3 expanding windows specified in contract v1.1 (`dev_folds()`).
